In [ ]:
import json
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd

# ======================== User configuration ========================
MULTI_RUN = Path("../outputs/simulation")
OUT_BASE = Path("../outputs/gain_curves")
BUDGET_GRID = ["1/20", "1/10", "3/20", "1/5", "1/4", "3/10", "7/20", "2/5", "9/20", "1/2",
               "11/20", "3/5", "13/20", "7/10", "3/4", "4/5", "17/20", "9/10", "19/20", "1"]
REFERENCE_BUDGET = "1/5"
REFERENCE_RULE = "population"
# ==========================================================

SETTING = ["split", "heatwave_type", "cooling_mode", "delta_t_C"]
CASE = SETTING + ["budget_share"]
GRID = [Fraction(b) for b in BUDGET_GRID]
MAIN_BUDGET = Fraction(REFERENCE_BUDGET)


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def resolve_run(path, marker):
    path = Path(path)
    require((path / marker).is_file(), f"{marker} not found in {path}")
    return path


def n_selected(budget, n):
    return int(-((-budget.numerator * int(n)) // budget.denominator)) if budget > 0 else 0


def expected_curve(scores, values):
    frame = pd.DataFrame(dict(score=np.asarray(scores, float), value=np.asarray(values, float)))
    require(np.isfinite(frame.to_numpy()).all(), "Ranking score or benefit is missing")
    frame["value"] = frame.groupby("score")["value"].transform("mean")
    ordered = frame.sort_values("score", ascending=False, kind="stable")["value"].to_numpy()
    return np.r_[0.0, np.cumsum(ordered)]


def run_gain_curves():
    run = resolve_run(MULTI_RUN, "multiyear_siting_efficiency.csv")
    status = json.loads((run / "summary.json").read_text(encoding="utf-8")).get("status")
    require(status == "completed", f"Multi-year simulation not completed: {run}")
    eff = pd.read_csv(run / "multiyear_siting_efficiency.csv")
    rules = sorted(eff.strategy.unique())
    cities = sorted(eff.city.unique())
    sim_budgets = sorted(eff.budget_share.unique())
    require(all(any(np.isclose(b, float(g)) for g in GRID) for b in sim_budgets), "BUDGET_GRID must include the budgets used in the simulation")
    require(MAIN_BUDGET in GRID, "The main budget must be in BUDGET_GRID")
    sim = eff.set_index(["city"] + CASE + ["strategy"])[["avoided_exposure", "n_treated"]]

    # ---- 1. For each city, scenario and rule: expected-benefit curve for selecting 0...n grid cells ----
    curves, meta = {}, {}
    for city in cities:
        cells = pd.read_csv(run / "by_city" / f"{city}_cell_responses.csv")
        require(all(f"score_{r}" in cells.columns for r in rules), f"{city}: grid-cell file lacks rule scores")
        for setting, c in cells.groupby(SETTING, sort=True):
            response = c.response_test.to_numpy(float)
            meta[(city,) + setting] = dict(n=len(c), baseline=float(c.baseline_exposure_test.iloc[0]))
            for rule in rules:
                scores = c[f"score_{rule}"].to_numpy(float)
                curves[(city,) + setting + (rule,)] = expected_curve(scores, response)
    settings = sorted({k[1:] for k in meta})

    # ---- 2. City results at the fixed budget levels ----
    rows, max_diff, n_checked = [], 0.0, 0
    for (city, *setting), m in meta.items():
        setting = tuple(setting)
        for budget in GRID:
            k = n_selected(budget, m["n"])
            for rule in rules:
                avoided = float(curves[(city,) + setting + (rule,)][k])
                rows.append(dict(zip(["city"] + SETTING, (city,) + setting)) | dict(
                    budget_share=float(budget), strategy=rule, n_eligible_cells=m["n"], n_treated=k,
                    avoided_exposure=avoided, baseline_exposure_test=m["baseline"]))
                match_sim = [b for b in sim_budgets if np.isclose(b, float(budget))]
                if match_sim:
                    ref = sim.loc[(city,) + setting + (match_sim[0], rule)]
                    require(int(ref.n_treated) == k, f"{city} {setting} {budget}: number of sited cells differs from the simulation")
                    max_diff = max(max_diff, abs(avoided - float(ref.avoided_exposure)))
                    n_checked += 1
    by_city = pd.DataFrame(rows)
    scale = float(eff.avoided_exposure.abs().max())
    require(n_checked == len(eff), f"Number of checked rows {n_checked} differs from the number of simulation rows {len(eff)}")
    require(max_diff <= 1e-9 * scale, f"Differs from the simulation output (max difference {max_diff:.3g})")
    pooled = by_city.groupby(CASE + ["strategy"], sort=True).agg(
        n_cities=("city", "nunique"), n_treated=("n_treated", "sum"), n_cells=("n_eligible_cells", "sum"),
        avoided=("avoided_exposure", "sum"), baseline=("baseline_exposure_test", "sum")).reset_index()
    pooled["actual_cell_share"] = pooled.n_treated / pooled.n_cells
    pooled["avoided_share_of_baseline"] = pooled.avoided / pooled.baseline

    # ---- 3. Exact whole-grid-cell curves: enumerate all siting allocations that arise as the nominal budget goes from 0 to 100% ----
    exact, match = [], []
    for setting in settings:
        ns = np.array([meta[(city,) + setting]["n"] for city in cities])
        baseline = sum(meta[(city,) + setting]["baseline"] for city in cities)
        bounds = sorted({Fraction(k, int(n)) for n in ns for k in range(1, int(n) + 1)})
        states = [(Fraction(0), np.zeros(len(ns), dtype=int))] + [
            (b, np.array([n_selected(b, n) for n in ns])) for b in bounds]
        require(all((b2[1] >= b1[1]).all() and b2[1].sum() > b1[1].sum() for b1, b2 in zip(states, states[1:])),
                "Siting allocations should increase strictly with the budget")
        main_k = np.array([n_selected(MAIN_BUDGET, n) for n in ns])
        target = sum(curves[(city,) + setting + (REFERENCE_RULE,)][k] for city, k in zip(cities, main_k))
        for rule in rules:
            city_curves = [curves[(city,) + setting + (rule,)] for city in cities]
            totals = np.array([sum(curve[k] for curve, k in zip(city_curves, ks)) for _, ks in states])
            for (b, ks), total in zip(states, totals):
                exact.append(dict(zip(SETTING, setting)) | dict(
                    strategy=rule, nominal_budget_fraction=str(b),
                    n_treated=int(ks.sum()), n_cells=int(ns.sum()), actual_cell_share=ks.sum() / ns.sum(),
                    avoided=float(total), baseline=baseline, avoided_share_of_baseline=total / baseline))
            reached = np.flatnonzero(totals >= target - 1e-9 * max(abs(target), 1.0))
            row = dict(zip(SETTING, setting)) | dict(
                strategy=rule, reference_budget=str(MAIN_BUDGET),
                reference_n_treated=int(main_k.sum()), n_cells=int(ns.sum()),
                target_avoided_share_of_baseline=target / baseline)
            if len(reached):
                b, ks = states[reached[0]]
                row |= dict(status="reached", nominal_budget_fraction=str(b),
                            n_treated_needed=int(ks.sum()), actual_cell_share_needed=ks.sum() / ns.sum(),
                            attained_avoided_share_of_baseline=totals[reached[0]] / baseline)
            else:
                row |= dict(status="not reached", nominal_budget_fraction="",
                            n_treated_needed=np.nan, actual_cell_share_needed=np.nan,
                            attained_avoided_share_of_baseline=np.nan)
            match.append(row)
    exact, match = pd.DataFrame(exact), pd.DataFrame(match)
    check = exact.merge(pooled[SETTING + ["strategy", "n_treated", "avoided"]], on=SETTING + ["strategy", "n_treated"],
                        suffixes=("", "_grid"))
    require(len(check) >= len(pooled) * 0.9 and np.allclose(check.avoided, check.avoided_grid, rtol=1e-12, atol=1e-6),
            "Exact curves do not match the summary at the fixed budget levels")

    out_dir = OUT_BASE
    out_dir.mkdir(parents=True, exist_ok=True)
    pooled.to_csv(out_dir / "gain_curves_pooled.csv", index=False)
    exact.to_csv(out_dir / "gain_curves_exact_pooled.csv", index=False)
    match.to_csv(out_dir / "cells_needed_to_match_reference.csv", index=False)
    (out_dir / "summary.json").write_text(json.dumps(dict(
        multiyear_run=str(run), budget_grid=BUDGET_GRID, rules=rules, reference_rule=REFERENCE_RULE,
        reference_budget=REFERENCE_BUDGET,
        rows_by_city=int(len(by_city)), rows_exact_curve=int(len(exact)),
        rows_checked_against_simulation=int(n_checked), max_abs_diff_against_simulation=max_diff),
        ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output:", out_dir)
    return out_dir


GAIN_OUT = run_gain_curves()